# 3. The PD model

This notebook builds the model that estimates each loan's probability of default (PD). It takes the features of a loan, such as the FICO score and income, and gives back one number between 0% and 100%, the chance that the loan will not be repaid. A lender can use that number to decide whether to approve the loan.

The model is built in three steps. Each loan's values are replaced by the WoE values found in notebook 2, the features that improve the predictions are chosen, and a logistic regression combines them into the PD. This is done separately for the 36-month and the 60-month loans, using the train sets only.

### 3.1 Applying the WoE values

Notebook 2 found the bins of each feature and the WoE of each bin. Here every loan's value is replaced by the WoE of the bin it falls in. 

In [1]:
from pathlib import Path
import pandas as pd
from optbinning import BinningProcess

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

train_36 = pd.read_parquet(PROCESSED / "train_36.parquet")
train_60 = pd.read_parquet(PROCESSED / "train_60.parquet")

binning_36 = BinningProcess.load(str(PROCESSED / "binning_36.pkl"))
binning_60 = BinningProcess.load(str(PROCESSED / "binning_60.pkl"))

features = [c for c in train_36.columns if c != "target"]

In [5]:
woe_36 = binning_36.transform(train_36[features], metric="woe")
woe_60 = binning_60.transform(train_60[features], metric="woe")

woe_36.head()

,purpose,verification_status,home_ownership,annual_inc,dti,delinq_2yrs,inq_last_6mths,open_acc,total_acc,pub_rec,revol_bal,revol_util,fico,credit_history_years,loan_to_income,emp_length_years
0,0.195011,0.112866,0.211858,-0.015751,0.109082,0.015269,-0.226903,-0.008336,0.100731,0.022113,-0.090669,0.188747,0.309707,0.177542,0.084528,0.005833
1,-0.047991,-0.044799,-0.183550,0.099124,-0.295136,0.015269,0.154944,-0.002175,0.100731,0.022113,-0.005317,-0.039651,-0.192948,0.171589,0.155567,0.111451
2,-0.047991,-0.044799,-0.183550,-0.111983,-0.411201,0.015269,-0.062767,-0.002175,-0.013288,0.022113,-0.005317,-0.216397,-0.115511,-0.009827,0.084528,-0.063137
3,-0.047991,-0.090945,0.211858,0.177458,-0.183842,0.015269,0.154944,0.014577,0.084620,0.022113,-0.090669,0.026180,-0.210793,0.003600,0.017391,0.111451
4,0.195011,-0.044799,-0.183550,0.177458,-0.295136,-0.030284,0.154944,-0.008336,0.041905,0.022113,0.242848,-0.121218,-0.346380,0.070203,-0.246843,0.111451


### 3.2 A first selection

The model chooses its own features with forward selection. It starts with no features and adds one at a time, each round taking the feature that improves the predictions the most.

Predictions are scored on loans the model was not fitted on. The train set is split into 5 parts, the model is fitted on 4 and scored on the 5th, and this is repeated so that every part is scored once. The score is the log-loss, which is lower when defaulted loans get a high PD and repaid loans get a low PD. A feature is added only if it improves the score in all 5 parts, and the selection stops when no remaining feature does.

This first selection is run on the whole train set of each group. It shows how the method works and how much each feature adds. The final features are settled in 3.3, where the selection is repeated on different samples of loans.

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

FOLDS = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def fold_losses(X, y):
    model = LogisticRegression(C=np.inf, max_iter=1000)
    return -cross_val_score(model, X, y, cv=FOLDS, scoring="neg_log_loss")


def forward_selection(woe, y):
    chosen = []
    steps = []
    best = fold_losses(np.zeros((len(y), 1)), y)
    print(f"No features: log-loss {best.mean():.5f}")

    while len(chosen) < woe.shape[1]:
        losses = {c: fold_losses(woe[chosen + [c]], y) for c in woe.columns if c not in chosen}
        passing = {c: l for c, l in losses.items() if (l < best).all()}
        if not passing:
            print("Stopped: no remaining feature improves all 5 parts")
            break
        winner = min(passing, key=lambda c: passing[c].mean())
        best = passing[winner]
        chosen.append(winner)
        steps.append({"feature": winner, "log_loss": best.mean()})
        print(f"Added {winner}: log-loss {best.mean():.5f}")

    return pd.DataFrame(steps)

In [7]:
steps_36 = forward_selection(woe_36, train_36["target"])
steps_36

No features: log-loss 0.38752
Added fico: log-loss 0.37997
Added annual_inc: log-loss 0.37642
Added inq_last_6mths: log-loss 0.37481
Added loan_to_income: log-loss 0.37348
Added purpose: log-loss 0.37234
Added dti: log-loss 0.37154
Added home_ownership: log-loss 0.37106
Added verification_status: log-loss 0.37093
Added credit_history_years: log-loss 0.37080
Added revol_bal: log-loss 0.37077
Added revol_util: log-loss 0.37072
Added open_acc: log-loss 0.37071
Stopped: no remaining feature improves all 5 parts


,feature,log_loss
0,fico,0.379967
1,annual_inc,0.376416
2,inq_last_6mths,0.374811
3,loan_to_income,0.373484
4,purpose,0.372340
5,dti,0.371537
6,home_ownership,0.371060
7,verification_status,0.370926
8,credit_history_years,0.370801
9,revol_bal,0.370772


In [8]:
steps_60 = forward_selection(woe_60, train_60["target"])
steps_60


No features: log-loss 0.56683
Added fico: log-loss 0.55687
Added loan_to_income: log-loss 0.55109
Added inq_last_6mths: log-loss 0.54915
Added purpose: log-loss 0.54751
Added revol_bal: log-loss 0.54713
Added revol_util: log-loss 0.54694
Added open_acc: log-loss 0.54678
Added total_acc: log-loss 0.54626
Added pub_rec: log-loss 0.54607
Stopped: no remaining feature improves all 5 parts


,feature,log_loss
0,fico,0.556869
1,loan_to_income,0.551091
2,inq_last_6mths,0.549149
3,purpose,0.547510
4,revol_bal,0.547133
5,revol_util,0.546944
6,open_acc,0.546776
7,total_acc,0.546262
8,pub_rec,0.546068


### 3.3 Checking that the selection is stable

A feature that really helps should be chosen whichever loans the model is built on. The selection is run 5 more times, each on a different random 80% of the train set, and we count how many times each feature is chosen. A feature chosen 5 times out of 5 is stable. One chosen only some of the time owes its place to chance.

In [10]:
import io
from contextlib import redirect_stdout


def stability(woe, y, runs=5):
    times_chosen = pd.Series(0, index=woe.columns)
    for run in range(runs):
        sample = woe.sample(frac=0.8, random_state=run).index
        with redirect_stdout(io.StringIO()):
            steps = forward_selection(woe.loc[sample], y.loc[sample])
        times_chosen[steps["feature"]] += 1
        print(f"Run {run + 1}: {len(steps)} features")
    return times_chosen.sort_values(ascending=False)

In [11]:
stability_36 = stability(woe_36, train_36["target"])
stability_36

Run 1: 11 features
Run 2: 13 features
Run 3: 11 features
Run 4: 10 features
Run 5: 12 features


purpose                 5
home_ownership          5
annual_inc              5
dti                     5
inq_last_6mths          5
total_acc               5
fico                    5
loan_to_income          5
verification_status     4
open_acc                4
pub_rec                 2
revol_util              2
credit_history_years    2
delinq_2yrs             1
revol_bal               1
emp_length_years        1
dtype: int64

In [12]:
stability_60 = stability(woe_60, train_60["target"])
stability_60

Run 1: 8 features
Run 2: 5 features
Run 3: 9 features
Run 4: 5 features
Run 5: 6 features


inq_last_6mths          5
fico                    5
loan_to_income          5
annual_inc              4
home_ownership          3
pub_rec                 3
purpose                 2
open_acc                2
revol_util              2
dti                     1
total_acc               1
verification_status     0
delinq_2yrs             0
revol_bal               0
credit_history_years    0
emp_length_years        0
dtype: int64

A feature is kept in the final model if it is chosen in at least 4 of the 5 runs. This allows for one unlucky run, while leaving out the features that are chosen only some of the time.

For the 60-month loans, two more features are added by judgement, `home_ownership` and `pub_rec`. Both are chosen in 3 of the 5 runs, just under the threshold, and both are well-known signs of credit risk. Home ownership says how settled a borrower is, and the 36-month model keeps it. Public records, such as court judgments and bankruptcies, show past trouble with debts. With far fewer loans in this group, a real effect can easily miss the threshold, so these two are kept.

In [13]:
final_36 = list(stability_36[stability_36 >= 4].index)
final_60 = list(stability_60[stability_60 >= 4].index) + ["home_ownership", "pub_rec"]

print("36-month:", final_36)
print("60-month:", final_60)

36-month: ['purpose', 'home_ownership', 'annual_inc', 'dti', 'inq_last_6mths', 'total_acc', 'fico', 'loan_to_income', 'verification_status', 'open_acc']
60-month: ['inq_last_6mths', 'fico', 'loan_to_income', 'annual_inc', 'home_ownership', 'pub_rec']


### 3.4 Fitting the models

A logistic regression is fitted on the final features of each group, the ones settled in 3.3.

- The weight, which says how much the log odds of default change when the feature's WoE goes up by one.
- The standard error, which says how uncertain the weight is. A different sample of loans would give a slightly different weight, and the standard error is the typical size of that difference.
- The p-value, which is the chance of getting a weight this far from zero if the feature really had no effect. A p-value below 0.05 is the usual sign that the effect is real.

Every weight should be negative, because a higher WoE means a safer bin, which should lower the PD. A positive weight is a warning that the feature repeats what another one already says.

In [15]:
import statsmodels.api as sm


def fit_model(woe, y, chosen):
    X = sm.add_constant(woe[chosen])
    return sm.Logit(y, X).fit(disp=False)


def weights_table(model):
    return pd.DataFrame({
        "weight": model.params.round(4),
        "std_error": model.bse.round(4),
        "p_value": model.pvalues.map("{:.2e}".format),
    })

In [16]:
model_36 = fit_model(woe_36, train_36["target"], final_36)
weights_table(model_36)

,weight,std_error,p_value
const,-1.8973,0.0053,0.00e+00
purpose,-0.9437,0.0360,3.02e-151
home_ownership,-0.4791,0.0297,1.14e-58
annual_inc,-0.5150,0.0220,1.63e-121
dti,-0.5971,0.0262,3.19e-115
inq_last_6mths,-1.0628,0.0305,2.25e-265
total_acc,-0.9612,0.0687,1.63e-44
fico,-0.8720,0.0154,0.00e+00
loan_to_income,-0.6854,0.0270,1.21e-142
verification_status,-0.6073,0.0609,2.06e-23


In [17]:
model_60 = fit_model(woe_60, train_60["target"], final_60)
weights_table(model_60)

,weight,std_error,p_value
const,-1.0774,0.0130,0.00e+00
inq_last_6mths,-1.0640,0.0890,5.70e-33
fico,-0.9094,0.0414,3.36e-107
loan_to_income,-0.8845,0.0586,1.78e-51
annual_inc,-0.3845,0.0762,4.48e-07
home_ownership,-0.6302,0.1037,1.23e-09
pub_rec,-0.5982,0.1458,4.10e-05


### 3.6 Checking for features that repeat each other

The VIF of a feature says how well it can be predicted from the other features in the model. A VIF of 1 means not at all. A VIF above 5 means the feature mostly repeats the others, which makes its weight unreliable.

In [18]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


def vif_table(woe, chosen):
    X = sm.add_constant(woe[chosen])
    vif = [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
    return pd.Series(vif, index=chosen).round(2)

In [21]:
vif_table(woe_36, final_36)

purpose                1.01
home_ownership         1.15
annual_inc             1.53
dti                    1.25
inq_last_6mths         1.03
total_acc              1.64
fico                   1.05
loan_to_income         1.26
verification_status    1.10
open_acc               1.46
dtype: float64

In [22]:
vif_table(woe_60, final_60)

inq_last_6mths    1.02
fico              1.04
loan_to_income    1.32
annual_inc        1.39
home_ownership    1.08
pub_rec           1.03
dtype: float64

### 3.6 Saving the models

The weights of the two models are saved so the next notebook can test them on the test sets. The weights, together with the bins from notebook 2, are all that is needed to work out the PD of any loan.

In [25]:
model_36.params.to_json(PROCESSED / "weights_36.json")
model_60.params.to_json(PROCESSED / "weights_60.json")